In [ ]:
import graphite


In [ ]:
dataset = graphite.Dataset(source='iam',
                           level='line',
                           training_ratio=None,
                           validation_ratio=None,
                           test_ratio=None,
                           lazy_mode=False,
                           seed=42)
print(dataset)


In [ ]:
augmentor = graphite.Augmentor(elastic_transform=[0.99, 43, 0.75],
                               erosion=[0.99, 11, 1],
                               dilation=[0.99, 3, 1],
                               mixup=[0.99, 0.3, 1],
                               perspective_transform=[0.99, 0.4],
                               salt_and_pepper=[0.99, 0.3],
                               gaussian_blur=[0.99, 11, 1],
                               shearing=[0.99, 30],
                               scaling=[0.99, 0.3],
                               rotation=[0.99, 3.0],
                               translation=[0.99, 0.3, 0.3],
                               seed=42)
print(augmentor)


In [ ]:
model = graphite.Model(network='flor', tokenizer=dataset.tokenizer, seed=42)
print(model)

model.compile(learning_rate=1e-3, run_index=None)


In [ ]:
training_data, training_steps = dataset.get_generator(partition='training', batch_size=16, augmentor=augmentor)
validation_data, validation_steps = dataset.get_generator(partition='validation', batch_size=16, augmentor=None)

history = model.fit(training_data=training_data,
                    training_steps=training_steps,
                    validation_data=validation_data,
                    validation_steps=validation_steps,
                    plateau_cooldown=0,
                    plateau_factor=0.2,
                    plateau_patience=10,
                    patience=20,
                    epochs=1000,
                    verbose=1)


In [ ]:
# <keras.callbacks.History at 0x2e298cc10>
history


In [ ]:
import numpy as np

print(np.array(predicts, dtype=object).shape)

print(len(predicts), len(predicts[0]), len(predicts[0][0]), len(predicts[0][0][0]))


In [ ]:
import numpy as np
import tensorflow as tf


test_data, test_steps = dataset.get_generator(partition='test', batch_size=16, augmentor=None)


def predict(test_data,
            test_steps,
            beam_width=100,
            top_paths=1,
            ctc_decode=True,
            token_decode=True,
            verbose=1):

    predicts = model.model.predict(x=test_data, steps=test_steps, verbose=verbose)
    decoded, probabilities = np.log(predicts.clip(min=1e-8)), np.array([])

    if ctc_decode:
        sequence_length = [predicts.shape[2]] * predicts.shape[0]
        decoded_paths, probabilities_list = [], []

        progbar = tf.keras.utils.Progbar(target=predicts.shape[1], unit_name='path_decode', verbose=verbose)

        for i in range(predicts.shape[1]):
            progbar.update(i)
            inputs = tf.transpose(predicts[:, i, :, :], perm=[1, 0, 2])

            decoded, log_probabilities = tf.nn.ctc_beam_search_decoder(inputs=inputs,
                                                                       sequence_length=sequence_length,
                                                                       beam_width=beam_width,
                                                                       top_paths=top_paths)

            probabilities_list.append(tf.exp(log_probabilities))

            decoded_pads = []
            for j in range(top_paths):
                sparse_decoded = tf.sparse.to_dense(decoded[j], default_value=-1)
                padding = [[0, 0], [0, predicts.shape[2] - tf.reduce_max(tf.shape(sparse_decoded)[1])]]
                decoded_pads.append(tf.pad(sparse_decoded, paddings=padding, constant_values=-1))

            decoded_paths.append(decoded_pads)
            progbar.update(i + 1)

        decoded = np.transpose(tf.stack(decoded_paths, axis=1), (0, 2, 1, 3))
        probabilities = np.transpose(tf.stack(probabilities_list, axis=1), (2, 0, 1))

        if token_decode:
            decoded_strings = []

            for i in range(decoded.shape[0]):
                instance_strings = []

                for j in range(decoded.shape[1]):
                    decoded_string = model.tokenizer.decode(decoded[i, j, :, :])
                    instance_strings.append(decoded_string)

                decoded_strings.append(instance_strings)

            decoded = np.array(decoded_strings, dtype=object)

    return decoded, probabilities


predicts, probabilities = predict(test_data=test_data,
                                  test_steps=test_steps,
                                  beam_width=15,
                                  top_paths=2,
                                  ctc_decode=True,
                                  token_decode=True,
                                  verbose=1)

print(predicts.shape, probabilities.shape)
print(predicts)


In [ ]:
# evaluate = evaluation.ocr_metrics(predicts=predicts,
#                                   ground_truth=ground_truth,
#                                   norm_accentuation=args.norm_accentuation,
#                                   norm_punctuation=args.norm_punctuation)
